In [ ]:
from typing import Callable
import numpy as np
import tqdm

import fit_force
import physics_model

In [39]:
class DummyForce(physics_model.TorqueLeverSimulation):
    def __init__(self):
        tt = np.linspace(0, 1, 500)
        super().__init__(lever_mass = 3.5e-3,    #kg
                         lever_length = 15e-2,   #meter
                         lever_range = (30, 100), #degrees
                         motor_baseline_torque = 3e-3,
                         motor_extra_torque = 2e-2,
                         motor_onset_angle = (75, 80),
                         friction_coeff = 0.0e-3,
                         dt=1e-3)
        self.dummy_force = -np.sin(4*tt)**2*tt - 0.1*tt**2
    
    def get_torque_rat(self):
        force = self.dummy_force[int(self.time / self.dt)]
        torque = force * self.lever_length * np.sin(self.theta) / 2
        return torque
df = DummyForce()
res = df.simulate_trial(duration=0.5)
ref_thetas = res["theta"]
ref_forces = df.dummy_force

In [40]:
jax_simulation = fit_force.TorqueLeverSimulationJAX(lever_mass = 3.5e-3,    #kg
                         lever_length = 15e-2,   #meter
                         lever_range = (30, 100), #degrees
                         motor_baseline_torque = 3e-3,
                         motor_extra_torque = 2e-2,
                         motor_onset_angle = (75, 80),
                         autoregressive_penalty=1e3,
                         friction_coeff = 0.0e-3,
                         dt=1e-3)

In [41]:
def step(theta, theta_dot, force):
    (next_theta, next_theta_dot), _ = jax_simulation.step((theta, theta_dot), force)
    return next_theta, next_theta_dot

In [42]:
class BinnedValue:
    
    def __init__(self, n_bins, min_val, max_val):
        self.n_bins = n_bins
        self.min_val = min_val
        self.max_val = max_val

    def bin2val(self, bin):
        return self.min_val + bin/self.n_bins * (self.max_val - self.min_val)
    
    def val2bin(self, val):
        if val < self.min_val or val > self.max_val:
            return -1
        else:
            return int((val - self.min_val) / (self.max_val - self.min_val) * self.n_bins)

In [ ]:
def fit_force(ref_thetas: np.Array, step_fcn: Callable):
    force_bins = BinnedValue(n_bins=100, min_val=-1, max_val=0)
    theta_dot_bins = BinnedValue(n_bins=100, min_val=-2, max_val=2)
    theta_offset_bins = BinnedValue(n_bins=20, min_val=-1, max_val=1)
    n_steps = len(ref_thetas)
    lowest_cost = np.full((n_steps, force_bins.n_bins, theta_dot_bins.n_bins, theta_offset_bins.n_bins), np.inf)
    lowest_cost[0] = 0.0
    backref = np.full((n_steps, force_bins.n_bins, theta_dot_bins.n_bins, theta_offset_bins.n_bins, 3), -1)
    for t in range(n_steps-1):
        for force_b in range(force_bins.n_bins):
            force = force_bins.bin2val(force_b)
            for theta_dot_b in range(theta_dot_bins.n_bins):
                theta_dot = theta_dot_bins.bin2val(theta_dot_b)
                for theta_offset_b in range(theta_offset_bins.n_bins):
                    theta_offset = theta_offset_bins.bin2val(theta_offset_b)
                    theta = ref_thetas[t] + theta_offset
                    for next_force_b in range(force_bins.n_bins):
                        if not np.isfinite(lowest_cost[t, force_b, theta_dot_b, theta_offset_b]):
                            continue
                        next_force = force_bins.bin2val(force_b)
                        next_theta, next_theta_dot = step_fcn(theta, theta_dot, force)
                        next_theta_offset_b = theta_offset_bins.val2bin(next_theta - ref_thetas[t+1])
                        next_theta_dot_b = theta_dot_bins.val2bin(next_theta_dot)
                        if next_theta_offset_b >= 0 and next_theta_dot_b >= 0:
                            cost  = lowest_cost[t, force_b, theta_dot_b, theta_offset_b]
                            cost += abs(force-next_force)
                            if cost < lowest_cost[t+1, next_force_b, next_theta_dot_b, next_theta_offset_b]:
                                lowest_cost[t+1, next_force_b, next_theta_dot_b, next_theta_offset_b] = cost
                                backref[t+1, next_force_b, next_theta_dot_b, next_theta_offset_b, 0] = force_b
                                backref[t+1, next_force_b, next_theta_dot_b, next_theta_offset_b, 1] = theta_dot_b
                                backref[t+1, next_force_b, next_theta_dot_b, next_theta_offset_b, 2] = theta_offset_b
    force_est = np.full(n_steps, np.nan)
    force_b, theta_dot_b, theta_offset_b = np.unravel_index(np.argmin(lowest_cost[-1]), lowest_cost.shape[1:])
    if force_b < 0 or theta_dot_b < 0 or theta_offset_b < 0:
        raise RuntimeError("Failed to find any compatible force sequence")
    for t in reversed(range(n_steps-1)):
        force_est[t] = force_bins.bin2val(force_b)
        force_b, theta_dot_b, theta_offset_b = backref[t, next_force_b, next_theta_dot_b, next_theta_offset_b]

Timestep 0, force 0:  24%|██▍       | 24/100 [02:02<06:28,  5.12s/it]


KeyboardInterrupt: 

In [14]:
backref.size  * 100 / 500

60000000.0

In [16]:
abs(-3)

3

In [36]:
np.unravel_index(np.argmin(np.random.normal(0, 1, (5,4,3,2))), (5,4,3,2))

(np.int64(4), np.int64(2), np.int64(2), np.int64(1))